In [ ]:
import torch
import torch.nn as nn
import math
import torch.nn.functional as F

In [ ]:
# Cell 2 - Fix Preprocessing (remove from model, use as external preprocessing)
import torch
import torchvision.transforms as transforms

class Preprocessing:
    def __init__(self, img_size=224, training=True):
        base_transforms = [
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ]
        
        if training:
            augmentations = [
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomRotation(degrees=15),
                transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05),
            ]
            self.transform = transforms.Compose(augmentations + base_transforms)
        else:
            self.transform = transforms.Compose(base_transforms)

    def __call__(self, img):
        # img should be PIL Image
        return self.transform(img)

In [3]:
class GhostConv(nn.Module):
    """
    Ghost Convolution module: generates more feature maps from intrinsic ones
    using cheap operations, inspired by the GhostNet paper.
    
    Args:
        inp (int): Number of input channels.
        oup (int): Number of output channels.
        kernel_size (int): Kernel size for primary convolution (default 1).
        ratio (int): Ratio for channels split between primary and cheap conv (default 2).
        dw_size (int): Kernel size for depthwise (cheap) convolution (default 3).
        stride (int): Stride for primary convolution (default 1).
        relu (bool): Whether to apply ReLU activations (default True).
    """
    def __init__(self, inp: int, oup: int, kernel_size: int = 1, ratio: int = 2, 
                 dw_size: int = 3, stride: int = 1, relu: bool = True):
        super(GhostConv, self).__init__()
        self.oup = oup
        assert kernel_size % 2 == 1, "Kernel size should be odd for symmetric padding"
        init_channels = math.ceil(oup / ratio)
        new_channels = init_channels * (ratio - 1)

        self.primary_conv = nn.Sequential(
            nn.Conv2d(inp, init_channels, kernel_size=kernel_size, stride=stride, padding=kernel_size//2, bias=False),
            nn.BatchNorm2d(init_channels),
            nn.ReLU(inplace=True) if relu else nn.Identity()
        )

        self.cheap_op = nn.Sequential(
            nn.Conv2d(init_channels, new_channels, kernel_size=dw_size, stride=1, padding=dw_size//2, 
                      groups=init_channels, bias=False),
            nn.BatchNorm2d(new_channels),
            nn.ReLU(inplace=True) if relu else nn.Identity()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.primary_conv(x)
        x2 = self.cheap_op(x1)
        out = torch.cat([x1, x2], dim=1)
        return out[:, :self.oup, :, :]


In [4]:
class FusedInvertedResidualBlock(nn.Module):
    """
    Fused Inverted Residual block:
    Combines expansion and depthwise convolutions into one fused conv for efficient computation,
    followed by a projection convolution to reduce channels.
    
    Args:
        inp (int): Number of input channels.
        oup (int): Number of output channels.
        stride (int): Stride for the first convolution (default 1).
        expand_ratio (int): Expansion factor for hidden dimension (default 4).
    """
    def __init__(self, inp: int, oup: int, stride: int = 1, expand_ratio: int = 4):
        super(FusedInvertedResidualBlock, self).__init__()
        self.stride = stride
        hidden_dim = int(round(inp * expand_ratio))
        self.use_res_connect = (self.stride == 1 and inp == oup)

        layers = []
        if expand_ratio != 1:
            # Fused conv expands channels, acts as first conv with kernel size 3
            layers.append(
                nn.Conv2d(inp, hidden_dim, kernel_size=3, stride=stride, padding=1, bias=False)
            )
            layers.append(nn.BatchNorm2d(hidden_dim))
            layers.append(nn.ReLU(inplace=True))
        else:
            # No expansion, keep hidden_dim same as inp
            hidden_dim = inp
        
        # Projection conv: reduces hidden_dim to oup
        layers.append(
            nn.Conv2d(hidden_dim, oup, kernel_size=1 if expand_ratio != 1 else 3, 
                      stride=1, padding=0 if expand_ratio != 1 else 1, bias=False)
        )
        layers.append(nn.BatchNorm2d(oup))

        self.block = nn.Sequential(*layers)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.use_res_connect:
            return x + self.block(x)
        else:
            return self.relu(self.block(x))


In [5]:
#Coordinate Attention
class HSigmoid(nn.Module):
    """Hard Sigmoid activation as per Coordinate Attention paper."""
    def __init__(self, inplace: bool = True):
        super(HSigmoid, self).__init__()
        self.relu6 = nn.ReLU6(inplace=inplace)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.relu6(x + 3) / 6

class HSwish(nn.Module):
    """Hard Swish activation using HSigmoid."""
    def __init__(self, inplace: bool = True):
        super(HSwish, self).__init__()
        self.hsigmoid = HSigmoid(inplace=inplace)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * self.hsigmoid(x)

class CoordAtt(nn.Module):
    """
    Coordinate Attention Block
    Args:
        inp (int): Number of input channels
        oup (int): Number of output channels
        reduction (int): Reduction ratio for intermediate channels
    """
    def __init__(self, inp: int, oup: int, reduction: int = 32):
        super(CoordAtt, self).__init__()
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))  # Pooling height, fixed width=1
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))  # Pooling width, fixed height=1

        mip = max(8, inp // reduction)  # Intermediate channels
        
        self.conv1 = nn.Conv2d(inp, mip, kernel_size=1, stride=1, padding=0)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = HSwish()
        
        self.conv_h = nn.Conv2d(mip, oup, kernel_size=1, stride=1, padding=0)
        self.conv_w = nn.Conv2d(mip, oup, kernel_size=1, stride=1, padding=0)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        n, c, h, w = x.size()
        
        # Aggregate spatial info along height and width separately
        x_h = self.pool_h(x)  # [n, c, h, 1]
        x_w = self.pool_w(x).permute(0, 1, 3, 2)  # [n, c, 1, w] -> [n, c, w, 1]
        
        # Concatenate and reduce channels
        y = torch.cat([x_h, x_w], dim=2)  # Concatenate along height dimension
        y = self.conv1(y)
        y = self.bn1(y)
        y = self.act(y)
        
        # Split to height and width attention tensors
        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)  # Transpose back
        
        # Generate attention maps and apply sigmoid
        a_h = self.conv_h(x_h).sigmoid()
        a_w = self.conv_w(x_w).sigmoid()
        
        # Apply attention maps multiplicatively
        out = identity * a_w * a_h
        
        return out


In [6]:
class PatchEmbedding(nn.Module):
    """
    Convert spatial feature map into patch tokens.
    
    Args:
        in_channels (int): Number of input channels.
        embed_dim (int): Embedding dimension of output tokens.
        patch_size (int): Size of patches (height and width).
    """
    def __init__(self, in_channels: int, embed_dim: int, patch_size: int = 16):
        super(PatchEmbedding, self).__init__()
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: (B, C, H, W)
        x = self.proj(x)            # (B, embed_dim, H/patch, W/patch)
        x = x.flatten(2)            # Flatten height and width to one dimension: (B, embed_dim, N)
        x = x.transpose(1, 2)       # Swap axes to (B, N, embed_dim)
        return x

class PositionalEncoding(nn.Module):
    """
    Add sinusoidal positional encoding to patch tokens.
    
    Args:
        embed_dim (int): Dimension of the embeddings.
        max_len (int): Maximum length of the sequence.
    """
    def __init__(self, embed_dim: int, max_len: int = 5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-torch.log(torch.tensor(10000.0)) / embed_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # Shape: (1, max_len, embed_dim)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: (B, N, embed_dim)
        x = x + self.pe[:, :x.size(1), :]
        return x


In [7]:
class LinearDifferentialAttention(nn.Module):
    """
    Linear Differential Attention (LDA) block as described in the DIFF Transformer paper.
    
    Args:
        embed_dim (int): Input embedding dimension.
        num_heads (int): Number of attention heads.
        dropout (float): Dropout rate.
        init (float): Initialization scalar constant controlling scaling of differential attention.
    """
    def __init__(self, embed_dim: int, num_heads: int = 8, dropout: float = 0.1, init: float = 0.8):
        super(LinearDifferentialAttention, self).__init__()
        assert embed_dim % num_heads == 0, "Embedding dimension must be divisible by number of heads"
        
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scaling = self.head_dim ** -0.5
        
        # Learnable scalar initialized as exp(init)
        self.alpha = nn.Parameter(torch.tensor(init).exp())
        
        # Linear projections for Q1, Q2, K1, K2, V as per the differential attention mechanism
        self.q_proj = nn.Linear(embed_dim, embed_dim * 2, bias=False)  # outputs Q1 and Q2 concatenated
        self.k_proj = nn.Linear(embed_dim, embed_dim * 2, bias=False)  # outputs K1 and K2 concatenated
        self.v_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
        
        self.norm = nn.GroupNorm(num_groups=num_heads, num_channels=embed_dim)  # Head-wise normalization
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (batch, seq_len, embed_dim)
        """
        B, N, C = x.shape
        
        # Normalize input
        x_norm = self.norm(x.transpose(1, 2)).transpose(1, 2)  # GN expects (B, C, N), reshape accordingly
        
        # Compute Q1, Q2, K1, K2 as splits of projections
        q = self.q_proj(x_norm)  # (B, N, 2 * C)
        k = self.k_proj(x_norm)  # (B, N, 2 * C)
        v = self.v_proj(x_norm)  # (B, N, C)
        
        Q1, Q2 = q.chunk(2, dim=-1)
        K1, K2 = k.chunk(2, dim=-1)
        
        # Reshape for multi-head attention
        Q1 = Q1.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)  # (B, heads, N, head_dim)
        Q2 = Q2.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        K1 = K1.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        K2 = K2.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        V = v.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Scaled dot-product attention scores
        scores1 = torch.matmul(Q1, K1.transpose(-2, -1)) * self.scaling  # (B, heads, N, N)
        scores2 = torch.matmul(Q2, K2.transpose(-2, -1)) * self.scaling
        
        # Apply softmax to both scores
        A1 = F.softmax(scores1, dim=-1)
        A2 = F.softmax(scores2, dim=-1)
        
        # Differential attention: difference between two attention maps
        attn = self.alpha * (A1 - A2)
        
        # Apply attention to values
        out = torch.matmul(attn, V)  # (B, heads, N, head_dim)
        
        # Concatenate heads and project output
        out = out.transpose(1, 2).contiguous().view(B, N, C)  # (B, N, embed_dim)
        out = self.out_proj(out)
        out = self.dropout(out)
        return out


In [8]:
import torch
import torch.nn as nn

class ResidualLayerNormBlock(nn.Module):
    """
    Residual block combined with Layer Normalization.
    Args:
        embed_dim (int): Embedding dimension of the tokens.
    """
    def __init__(self, embed_dim: int):
        super(ResidualLayerNormBlock, self).__init__()
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x: torch.Tensor, residual: torch.Tensor = None) -> torch.Tensor:
        """
        x: (batch, seq_len, embed_dim) - input tensor (usually the output from LDA)
        residual: (batch, seq_len, embed_dim) - tensor to add as skip connection. If None, use input x as residual.
        """
        if residual is None:
            residual = x
        x = self.norm(x)
        return x + residual


In [9]:
class BottleneckFFN(nn.Module):
    """
    Bottleneck Feed Forward Network used in transformer variants.
    
    Args:
        inp (int): Input feature dimension (embedding dimension).
        oup (int): Output feature dimension.
        bottleneck_ratio (float): Reduction ratio for bottleneck hidden dimension.
        dropout (float): Dropout rate.
    """
    def __init__(self, inp: int, oup: int, bottleneck_ratio: float = 0.25, dropout: float = 0.1):
        super(BottleneckFFN, self).__init__()
        bottleneck_channels = max(1, int(inp * bottleneck_ratio))
        
        self.fc1 = nn.Linear(inp, bottleneck_channels)
        self.norm1 = nn.LayerNorm(bottleneck_channels)
        self.act = nn.GELU()
        self.dropout1 = nn.Dropout(dropout)
        
        self.fc2 = nn.Linear(bottleneck_channels, oup)
        self.norm2 = nn.LayerNorm(oup)
        self.dropout2 = nn.Dropout(dropout)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.norm1(x)
        x = self.act(x)
        x = self.dropout1(x)
        
        x = self.fc2(x)
        x = self.norm2(x)
        x = self.dropout2(x)
        
        return x


In [10]:
class GlobalAveragePooling(nn.Module):
    """
    Global Average Pooling over the sequence length dimension.
    Input shape: (batch, seq_len, embed_dim)
    Output shape: (batch, embed_dim)
    """
    def __init__(self):
        super(GlobalAveragePooling, self).__init__()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Average across seq_len dimension (dim=1)
        return x.mean(dim=1)


In [11]:
class ClassifierHead(nn.Module):
    """
    Final classification block with linear layer and softmax activation.
    
    Args:
        embed_dim (int): Input feature dimension.
        num_classes (int): Number of output classes.
    """
    def __init__(self, embed_dim: int, num_classes: int):
        super(ClassifierHead, self).__init__()
        self.fc = nn.Linear(embed_dim, num_classes)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: (batch, embed_dim)
        logits = self.fc(x)
        probs = F.softmax(logits, dim=-1)
        return probs


In [ ]:
# Cell 12 - Fix MobilePlantViTModel
class MobilePlantViTModel(nn.Module):
    def __init__(self, 
                 img_size=224,
                 num_classes=1000,
                 ghost_conv_params={},
                 fused_ir_params={},
                 coord_att_params={},
                 patch_embed_params={},
                 pos_encoding_params={},
                 lda_params={},
                 res_ln_params={},
                 bottleneck_ffn_params={},
                 ):
        super(MobilePlantViTModel, self).__init__()
        
        # Remove preprocessing from model - it should be done externally
        self.ghost_conv = GhostConv(**ghost_conv_params)
        self.fused_inverted_residual = FusedInvertedResidualBlock(**fused_ir_params)
        self.coord_attention = CoordAtt(**coord_att_params)
        self.patch_embedding = PatchEmbedding(**patch_embed_params)
        self.positional_encoding = PositionalEncoding(**pos_encoding_params)
        self.lda = LinearDifferentialAttention(**lda_params)
        self.res_layer_norm = ResidualLayerNormBlock(**res_ln_params)
        self.bottleneck_ffn = BottleneckFFN(**bottleneck_ffn_params)
        self.global_avg_pool = GlobalAveragePooling()
        self.classifier = ClassifierHead(embed_dim=bottleneck_ffn_params['oup'], num_classes=num_classes)
        
    def forward(self, x):
        # x should already be preprocessed tensor (B, 3, H, W)
        x = self.ghost_conv(x)
        x = self.fused_inverted_residual(x)
        x = self.coord_attention(x)
        x = self.patch_embedding(x)
        x = self.positional_encoding(x)
        x = self.lda(x)
        x = self.res_layer_norm(x)
        x = self.bottleneck_ffn(x)
        x = self.global_avg_pool(x)
        x = self.classifier(x)
        return x